# ReBRAC Broad Validation — Colab Driver

Spec: `docs/superpowers/specs/2026-05-04-rebrac-broad-validation-design.md`
Plan: `docs/superpowers/plans/2026-05-04-rebrac-broad-validation-plan.md`

This notebook drives 4 Colab Pro / L4 sessions:

- **S1**: collect 7 new datasets (~6 h)
- **S2**: P1 training — 7 ReBRAC spokes × 2 seeds + A2 TD3+BC × 2 seeds (~8 h)
- **S3**: conditional P2 — β refit + 5-seed extension on triggered spokes (~5–8 h)
- **S4**: buffer / analysis (~2 h)

Run cells sequentially; each cell is `[skip]`-resume safe via `transitions.npz` /
`trainer_state.json` / `agent_final.pt` existence checks.


In [ ]:
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5")
except ModuleNotFoundError:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
print("git rev:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("git status:")
print(subprocess.check_output(["git", "status", "--short"], text=True))


In [ ]:
import subprocess
from scripts.broad_validation_spoke_registry import REGISTRY

SINGLE_POLICY_SPOKES = ["A1", "A3", "B1", "B2", "C1", "C3"]
for spoke_id in SINGLE_POLICY_SPOKES:
    cfg = REGISTRY[spoke_id]
    out_dir = Path("offline_data") / cfg.dataset_name
    if (out_dir / "transitions.npz").exists():
        print(f"[skip] {spoke_id}: dataset exists")
        continue
    cmd = [
        "python", "-m", "scripts.collect_offline_data",
        "--policy", cfg.collector_policy,
        "--flow", cfg.flow_path,
        "--probe-layout", cfg.probe_layout,
        "--task-geometry", cfg.task_geometry,
        "--target-speed", str(cfg.target_speed),
        "--history-length", "4",
        "--objective", "efficiency_v2",
        "--episodes", "1000",
        "--seed", "0",
        "--num-workers", "8",
        "--output-dir", str(out_dir),
    ]
    print(f"[collect] {spoke_id}:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
import subprocess
from pathlib import Path
from scripts.broad_validation_spoke_registry import REGISTRY

cfg = REGISTRY["A2"]
final_dir = Path("offline_data") / cfg.dataset_name

if (final_dir / "transitions.npz").exists():
    print(f"[skip] A2 mix already exists at {final_dir}")
else:
    sub_a = Path("offline_data") / "mix_tmp_goalseek_500ep_seed0"
    sub_b = Path("offline_data") / "mix_tmp_crosscomp_500ep_seed1"
    common = [
        "--flow", cfg.flow_path,
        "--probe-layout", cfg.probe_layout,
        "--task-geometry", cfg.task_geometry,
        "--target-speed", str(cfg.target_speed),
        "--history-length", "4",
        "--objective", "efficiency_v2",
        "--episodes", "500",
        "--num-workers", "8",
    ]
    if not (sub_a / "transitions.npz").exists():
        subprocess.run(
            ["python", "-m", "scripts.collect_offline_data",
             "--policy", "goalseek", "--seed", "0",
             "--output-dir", str(sub_a), *common],
            check=True,
        )
    if not (sub_b / "transitions.npz").exists():
        subprocess.run(
            ["python", "-m", "scripts.collect_offline_data",
             "--policy", "crosscomp", "--seed", "1",
             "--output-dir", str(sub_b), *common],
            check=True,
        )
    subprocess.run(
        ["python", "-m", "scripts.concat_offline_datasets",
         "--input-dir", str(sub_a),
         "--input-dir", str(sub_b),
         "--output-dir", str(final_dir),
         "--mix-strategy", "episode_level",
         "--task-sampler", "anchor_distribution"],
        check=True,
    )
    print(f"[done] A2 mix at {final_dir}")


In [ ]:
import subprocess
from pathlib import Path
from scripts.broad_validation_spoke_registry import REGISTRY

UNIQUE_DATASETS = {
    cfg.dataset_name: cfg.probe_layout
    for cfg in REGISTRY.values()
}
for dataset_name, probe in UNIQUE_DATASETS.items():
    dataset_dir = Path("offline_data") / dataset_name
    if not (dataset_dir / "transitions.npz").exists():
        print(f"[warn] missing dataset: {dataset_dir}")
        continue
    subprocess.run(
        ["python", "-m", "scripts.write_sanity_card",
         "--dataset-dir", str(dataset_dir),
         "--expected-probe-layout", probe],
        check=True,
    )


In [ ]:
import os, subprocess
from scripts.broad_validation_spoke_registry import REGISTRY

P1_ORDER = ["A1", "A2", "A2-td3bc", "A3", "B1", "B2", "C1", "C3"]
for spoke_id in P1_ORDER:
    env = os.environ.copy()
    env["SPOKE_ID"] = spoke_id
    env["PHASE"] = "p1"
    env["DEVICE"] = "cuda"
    print(f"\n========== P1: {spoke_id} ==========")
    subprocess.run(
        ["bash", "scripts/run_offline_rebrac_broad.sh"],
        env=env, check=True,
    )


In [ ]:
import subprocess
subprocess.run(
    ["python", "-m", "scripts.summarize_broad_validation"],
    check=True,
)


In [ ]:
import json
from pathlib import Path

decisions_path = Path("results/offline/rebrac/broad_validation/summaries/trigger_decisions.json")
decisions = json.loads(decisions_path.read_text())
triggered = [d for d in decisions if d["triggered"]]
print(f"Triggered spokes ({len(triggered)}):")
for d in triggered:
    print(f"  {d['spoke_id']}: delta_pp={d['delta_pp']:+.3f} reasons={d['reasons']}")
print()
print("Untriggered spokes:")
for d in decisions:
    if not d["triggered"]:
        print(f"  {d['spoke_id']}: delta_pp={d['delta_pp']:+.3f}")


## P2 deepening (conditional)

Edit cell 10 below: set `TRIGGERED_SPOKES` to the spoke IDs from cell 8 that
need β refit. The β refit runs (β1=2, β2=2) and (β1=4, β2=1) on seed 42 only
(spec §6.2). After cell 10, inspect `summaries/p1_overview.csv` to determine
the winner. Edit cell 11 with the winner per spoke, then run cell 11 to extend
to 5 seeds.

If `len(triggered) == 0`, skip cells 10 and 11.


In [ ]:
import os, subprocess

# EDIT THIS LIST after running cell 8.
TRIGGERED_SPOKES: list[str] = []  # e.g. ["A1", "C3"]

for spoke_id in TRIGGERED_SPOKES:
    for actor_b, critic_b in [("2.0", "2.0"), ("4.0", "1.0")]:
        env = os.environ.copy()
        env.update({
            "SPOKE_ID": spoke_id,
            "PHASE": "p2_refit",
            "ACTOR_PENALTY_COEFS": actor_b,
            "CRITIC_PENALTY_COEFS": critic_b,
        })
        print(f"\n========== P2 refit: {spoke_id} (β1={actor_b}, β2={critic_b}) ==========")
        subprocess.run(
            ["bash", "scripts/run_offline_rebrac_broad.sh"],
            env=env, check=True,
        )


In [ ]:
import os, subprocess

# EDIT THIS DICT after inspecting refit results.
# Winner format: {"A1": ("4.0", "2.0"), "C3": ("2.0", "2.0"), ...}
WINNERS: dict[str, tuple[str, str]] = {}

for spoke_id, (actor_b, critic_b) in WINNERS.items():
    env = os.environ.copy()
    env.update({
        "SPOKE_ID": spoke_id,
        "PHASE": "p2_5seed",
        "ACTOR_PENALTY_COEFS": actor_b,
        "CRITIC_PENALTY_COEFS": critic_b,
    })
    print(f"\n========== P2 5-seed: {spoke_id} (β1={actor_b}, β2={critic_b}) ==========")
    subprocess.run(
        ["bash", "scripts/run_offline_rebrac_broad.sh"],
        env=env, check=True,
    )


In [ ]:
import subprocess
subprocess.run(
    ["python", "-m", "scripts.summarize_broad_validation"],
    check=True,
)


In [ ]:
import subprocess
subprocess.run(["git", "add",
                "results/offline/rebrac/broad_validation/summaries/p1_overview.csv",
                "results/offline/rebrac/broad_validation/summaries/p1_overview.json",
                "results/offline/rebrac/broad_validation/summaries/trigger_decisions.json"],
               check=True)
print(subprocess.check_output(["git", "status", "--short"], text=True))
